# ClusterBP v2: другие модели последовательностей вместо GRU

Почему в последовательной ветке стоит GRU? Сравнение с тремя альтернативами на тех же входах (20 последних измерений, 5 каналов) и в той же схеме обучения:

- **TimeAwareGRU** — GRU с затуханием скрытого состояния по времени между измерениями (идея GRU-D, Che et al. 2018, и DATA-GRU, Tan et al. 2020: учёт нерегулярности ряда);
- **TCN** — свёрточная сеть с причинными расширенными свёртками;
- **Transformer** — небольшой энкодер с вниманием по всем измерениям окна.

Каждая модель сравнивается с GRU и отдельно, и в стекинге с XGB_full (как итоговая модель ClusterBP v2): важно, меняет ли более сложная последовательная модель качество всего ансамбля.
Фолды, внутренняя валидация, прореживание отрицательных и ранняя остановка — как в `clusterbp_v2.ipynb`. GRU и XGB_full берутся из уже обученных моделей фолдов.

In [1]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

from clusterbp_lib import (RESULTS_DIR, SEED, N_FOLDS, FEATURE_SETS, SEQ_MODELS, load_cached, fold_masks,
                           add_phenotype_columns, load_gru, load_xgb, train_seq_model, gru_logits, sigmoid, logit,
                           best_threshold, boot_eval, ci)

warnings.filterwarnings('ignore')
torch.set_num_threads(os.cpu_count())
SEQ_TARGETS = ['A', 'B']
NEW = ['TimeAwareGRU', 'TCN', 'Transformer']

win, X_seq, seq_len, pheno_win, oof = load_cached()
pid = win['pid'].values

In [2]:
store = {t: {} for t in SEQ_TARGETS}
info = []


def put(t, name, te, s, p):
    st = store[t].setdefault(name, [np.full(len(win), np.nan), np.zeros(len(win), np.int8)])
    st[0][te] = s[te]
    st[1][te] = p[te]


t_all = time.time()
for k in range(N_FOLDS):
    tr, va, te = fold_masks(win, k)
    add_phenotype_columns(win, pheno_win, k)
    for t in SEQ_TARGETS:
        y = win[f'y_{t}'].values
        px = load_xgb('full', t, k).predict_proba(win[FEATURE_SETS['full']].values)[:, 1]
        gm, mu, sd, corr = load_gru(t, k)
        seq_probs = {'GRU': sigmoid(gru_logits(gm, X_seq, seq_len, mu, sd) + corr)}
        for name in NEW:
            t0 = time.time()
            m, mu_, sd_, corr_, log = train_seq_model(SEQ_MODELS[name], X_seq[tr], seq_len[tr], y[tr],
                                                      X_seq[va], seq_len[va], y[va], SEED + k)
            seq_probs[name] = sigmoid(gru_logits(m, X_seq, seq_len, mu_, sd_) + corr_)
            info.append({'target': t, 'fold': k, 'model': name, 'эпох': len(log),
                         'параметров': sum(p.numel() for p in m.parameters()), 'секунд': time.time() - t0})
        for name, ps in seq_probs.items():
            thr, _ = best_threshold(y[va], ps[va])
            put(t, name, te, ps, (ps > thr).astype(np.int8))
            st = LogisticRegression(max_iter=3000).fit(np.column_stack([logit(ps[va]), logit(px[va])]), y[va])
            s = st.predict_proba(np.column_stack([logit(ps), logit(px)]))[:, 1]
            thr, _ = best_threshold(y[va], s[va])
            put(t, f'{name} + XGB_full (стекинг)', te, s, (s > thr).astype(np.int8))
    print(f'Фолд {k}: {(time.time() - t_all) / 60:.1f} мин')
pd.DataFrame(info).to_csv(os.path.join(RESULTS_DIR, 'sequence_models_training.csv'), index=False)

Фолд 0: 62.9 мин
Фолд 1: 128.9 мин
Фолд 2: 196.1 мин
Фолд 3: 351.7 мин
Фолд 4: 413.3 мин


## Результаты

Разности — относительно GRU (для отдельных моделей) и относительно стекинга GRU + XGB_full (для ансамблей); 95% ДИ — бутстреп по пациентам.

In [3]:
rows = []
for t in SEQ_TARGETS:
    y = win[f'y_{t}'].values.astype(float)
    labels = list(store[t])
    models = [(store[t][n][0], store[t][n][1].astype(float)) for n in labels]
    B = boot_eval(y, pid, models, n_boot=1000)
    for j, lab in enumerate(labels):
        ref = 'GRU + XGB_full (стекинг)' if 'стекинг' in lab else 'GRU'
        jr = labels.index(ref)
        d = B[:, j, 1] - B[:, jr, 1]
        rows.append({'target': t, 'модель': lab, 'AUROC': roc_auc_score(y, models[j][0]),
                     'AUPRC': average_precision_score(y, models[j][0]), 'AUPRC 95% ДИ': ci(B[:, j, 1]),
                     'macro_F1': f1_score(y, models[j][1], average='macro'),
                     'ΔAUPRC к ' + ref: '' if lab == ref else f'{np.mean(d):+.4f} {ci(d)}'})
seq = pd.DataFrame(rows)
seq.to_csv(os.path.join(RESULTS_DIR, 'sequence_models.csv'), index=False)
pd.set_option('display.max_colwidth', 60)
display(seq.round(4))
display(pd.DataFrame(info).groupby(['target', 'model'])[['эпох', 'параметров', 'секунд']].mean().round(1))

,target,модель,AUROC,AUPRC,AUPRC 95% ДИ,macro_F1,ΔAUPRC к GRU,ΔAUPRC к GRU + XGB_full (стекинг)
0,A,GRU,0.9073,0.4336,[0.404; 0.462],0.7142,,NaN
1,A,GRU + XGB_full (стекинг),0.9261,0.4905,[0.461; 0.519],0.7353,NaN,
2,A,TimeAwareGRU,0.9082,0.4311,[0.400; 0.459],0.7139,-0.0025 [-0.006; 0.001],NaN
3,A,TimeAwareGRU + XGB_full (стекинг),0.9262,0.4906,[0.461; 0.519],0.7353,NaN,+0.0000 [-0.000; 0.001]
4,A,TCN,0.9074,0.4303,[0.400; 0.459],0.7159,-0.0034 [-0.009; 0.002],NaN
5,A,TCN + XGB_full (стекинг),0.9263,0.4914,[0.462; 0.520],0.7351,NaN,+0.0008 [0.000; 0.002]
6,A,Transformer,0.9094,0.4300,[0.400; 0.457],0.7154,-0.0037 [-0.010; 0.002],NaN
7,A,Transformer + XGB_full (стекинг),0.9264,0.4911,[0.461; 0.520],0.7351,NaN,+0.0005 [-0.000; 0.001]
8,B,GRU,0.9206,0.6659,[0.649; 0.683],0.7849,,NaN
9,B,GRU + XGB_full (стекинг),0.9358,0.7138,[0.697; 0.730],0.8019,NaN,


эпох  параметров  секунд
target model                                 
A      TCN            9.6     49921.0   777.8
       TimeAwareGRU  25.0     18050.0   164.1
       Transformer   10.4     68673.0   206.5
B      TCN           12.0     49921.0  1820.3
       TimeAwareGRU  19.0     18050.0  1498.3
       Transformer    9.8     68673.0   469.3